# 01 - Raw Data Audit

Audit raw sources, validate schemas, and record data quality notes.


Before analyzing churn, we need to validate whether the raw winer DTC data is reliable enough to support retention analysis.

This notebook audits:
- Table structure
- Row counts
- Primary key uniquness
- Foreign key relationships
- Missing values
- Date ranges
- Member age elligibility
- Basic source-system quality issues

The goal for this analysis is to understand whether the source data can answer why members churn.

In [25]:
# Load libraries and data

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)

RAW_DIR = Path('../data/raw')
DIM_DIR = Path('../data/dim')



raw_files = {
    'raw_club_releases': 'raw_club_releases.csv',
    'raw_email_events': 'raw_email_events.csv',
    'raw_event_attendance': 'raw_event_attendance.csv',
    'raw_members': 'raw_members.csv',
    'raw_membership_events': 'raw_membership_events.csv',
    'raw_order_items': 'raw_order_items.csv',
    'raw_orders': 'raw_orders.csv',
    'raw_release_wines': 'raw_release_wines.csv',
    'raw_tasting_room_visits': 'raw_tasting_room_visits.csv',
    'raw_wines': 'raw_wines.csv'
}

dim_files = {
   'dim_acquisition_sources': 'dim_acquisition_sources.csv',
   'dim_club_tiers': 'dim_club_tiers.csv',
   'dim_dates': 'dim_dates.csv',
   'dim_members': 'dim_members.csv',
   'dim_releases': 'dim_releases.csv',
   'dim_wines': 'dim_wines.csv'
}

raw = {
    name:pd.read_csv(RAW_DIR/file)
    for name, file in raw_files.items()
}

dims = {
    name:pd.read_csv(DIM_DIR/file)
    for name, file in dim_files.items()
}


In [20]:
# Validate Row Counts

row_counts = []

for name, df in raw.items():
    row_counts.append({
        'table_name': name,
        'layer': 'raw',
        'row_count': len(df),
        'column_count': df.shape[1]
    })

for name, df in dims.items():
    row_counts.append({
        'table_name': name,
        'layer': 'dimension',
        'row_count': len(df),
        'column_count': df.shape[1]
    })

row_counts_df = pd.DataFrame(row_counts).sort_values(['layer', 'table_name'])
row_counts_df

,table_name,layer,row_count,column_count
15,dim_wines,dimension,55,17
10,dim_acquisition_sources,dimension,8,7
11,dim_club_tiers,dimension,4,10
12,dim_dates,dimension,1826,13
13,dim_members,dimension,4200,22
14,dim_releases,dimension,20,17
0,raw_club_releases,raw,20,13
1,raw_email_events,raw,223836,10
2,raw_event_attendance,raw,9634,11
3,raw_members,raw,4200,20


The dataset contains enough behavioral depth to support retention analysis and analyze churn as a relationship and behavior problem.

In [21]:
# Inspect schemas

for name, df in raw.items():
    print(f'\n{name}')
    print('-' * len(name))
    print(df.dtypes)


raw_club_releases
-----------------
release_id                 object
release_name               object
release_year                int64
release_season             object
release_open_date          object
release_ship_date          object
customization_deadline     object
default_2_bottle_value      int64
default_4_bottle_value      int64
default_6_bottle_value      int64
default_12_bottle_value     int64
release_theme              object
created_at                 object
dtype: object

raw_email_events
----------------
email_event_id    object
member_id         object
campaign_id       object
send_date         object
email_type        object
event_type        object
link_category     object
release_id        object
device_type       object
created_at        object
dtype: object

raw_event_attendance
--------------------
event_attendance_id          object
member_id                    object
event_date                   object
event_name                   object
event_type           

*** will need to change date fields to date format before building fact tables ***

In [26]:
# Drop Unnecessary Columns

# Columns to drop from raw tables
raw_columns_to_drop = {
    "raw_members": [
        "age_verified_date",
        "age_verification_method",
        "age_verified_flag",
        "created_at",
        "updated_at"
    ],
    "raw_membership_events": [
        "created_at"
    ],
    "raw_email_events": [
        "created_at"
    ],
    "raw_tasting_room_visits": [
        "created_at"
    ],
    "raw_event_attendance": [
        "created_at"
    ],
    "raw_wines": [
        "sellout_date",
        "created_at"
    ],
    "raw_club_releases": [
        "created_at"
    ],
    "raw_release_wines": [
        "created_at"
    ]
}

# Columns to drop from dimension tables
dim_columns_to_drop = {
    "dim_members": [
        "age_verified_date",
        "age_verification_method",
        "age_verified_flag",
        "created_at",
        "updated_at"
    ],
    "dim_wines": [
        "sellout_date"
    ],
    "dim_releases": [
        "created_at"
    ]
}

# Drop columns safely
for table_name, columns in raw_columns_to_drop.items():
    raw[table_name] = raw[table_name].drop(
        columns=columns,
        errors="ignore"
    )

for table_name, columns in dim_columns_to_drop.items():
    dims[table_name] = dims[table_name].drop(
        columns=columns,
        errors="ignore"
    )

In [27]:
# Confirm dropped columns were removed

for table_name, columns in raw_columns_to_drop.items():
    remaining_cols = [col for col in columns if col in raw[table_name].columns]
    print(f"{table_name}: {remaining_cols}")

for table_name, columns in dim_columns_to_drop.items():
    remaining_cols = [col for col in columns if col in dims[table_name].columns]
    print(f"{table_name}: {remaining_cols}")

raw_members: []
raw_membership_events: []
raw_email_events: []
raw_tasting_room_visits: []
raw_event_attendance: []
raw_wines: []
raw_club_releases: []
raw_release_wines: []
dim_members: []
dim_wines: []
dim_releases: []


In [ ]:
# Save updaed datasets to our df

for table_name, df in raw.items():
    df.to_csv(RAW_DIR / f"{table_name}.csv", index=False)

for table_name, df in dims.items():
    df.to_csv(DIM_DIR / f"{table_name}.csv", index=False)

In [30]:
# Parse Key Dates

date_columns = {
    'raw_members': ['join_date', 'birth_date'],
    'raw_orders': ['order_date'],
    'raw_order_items': ['created_at'],
    'raw_membership_events': ['event_date'],
    'raw_tasting_room_visits': ['visit_date'],
    'raw_event_attendance': ['event_date'],
    'raw_wines': ['release_date'],
    'raw_club_releases': ['release_open_date', 'release_ship_date', 'customization_deadline']
}

for table_name, cols in date_columns.items():
    for col in cols:
        if col in raw[table_name].columns:
            raw[table_name][col] = pd.to_datetime(raw[table_name][col], errors='coerce')

In [ ]:
# Primary Key Checks
# Valid primary keys should not have any nulls, duplicates, one unique identifier per row



